In [1]:
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy.sql import text
import warnings
import sys
import os # Esto sube un nivel en las carpetas para encontrar la raíz del proyecto
from dotenv import load_dotenv

sys.path.append(os.path.abspath(".."))

import src.db_builder as db

# Verificar dónde está buscando el .env
print("Directorio actual:", os.getcwd())

# Cargar y comprobar variables
load_dotenv("../.env")

Directorio actual: c:\Users\dacil\Desktop\Adalab\proyectos\Proyectos\e-commerce-operations-insights\notebooks


True

In [2]:
warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None) # para poder visualizar todas las columnas de los DataFrames

In [3]:
df_data = pd.read_csv("../files/processed/dataco_supply_chain.csv", sep=None, engine="python", encoding="latin-1")

In [4]:
df_data.sample(5)

,type,days_for_shipping_real,days_for_shipment_scheduled,benefit_per_order,sales_per_customer,delivery_status,late_delivery_risk,category_id,category_name,customer_city,customer_country,customer_fname,customer_id,customer_lname,customer_segment,customer_state,customer_street,customer_zipcode,department_id,department_name,latitude,longitude,order_city,order_country,order_customer_id,order_id,order_item_cardprod_id,order_item_discount,order_item_discount_rate,order_item_id,order_item_product_price,order_item_profit_ratio,order_item_quantity,sales,order_item_total,order_profit_per_order,order_region,order_status,product_card_id,product_category_id,product_name,product_price,shipping_mode,delivery_date,order_date,order_time
138238,CASH,2,4,125.73,371.98,Advance shipping,0,45,Fishing,Caguas,Puerto Rico,Mary,9368,Cabrera,Consumer,PR,7464 Quaking Stead,725,7,Fan Shop,18.225262,-66.370529,Carrefour,HaitÃ­,9368,56961,1004,28.00,0.07,142465,399.98,0.34,1,399.98,371.98,125.73,Caribbean,CLOSED,1004,45,Field & Stream Sportsman 16 Gun Fire Safe,399.98,Standard Class,04-13-2017,04-11-2017,11:37
157755,TRANSFER,3,2,41.74,123.49,Late delivery,1,18,Men's Footwear,Caguas,Puerto Rico,Andrea,8528,Maldonado,Consumer,PR,2143 Colonial Cloud Dale,725,4,Apparel,18.215502,-66.370590,SÃ£o Paulo,Brasil,8528,51297,403,6.50,0.05,128194,129.99,0.34,1,129.99,123.49,41.74,South America,PENDING,403,18,Nike Men's CJ Elite 2 TD Football Cleat,129.99,Second Class,01-21-2017,01-18-2017,19:16
105203,TRANSFER,6,4,13.19,43.98,Late delivery,1,46,Indoor/Outdoor Games,Phoenix,EE. UU.,James,3941,Harrell,Consumer,AZ,1338 Indian Prairie Extension,85040,7,Fan Shop,33.403561,-112.021400,AÃ§u,Brasil,3941,61410,1014,6.00,0.12,153597,49.98,0.30,1,49.98,43.98,13.19,South America,PROCESSING,1014,46,O'Brien Men's Neoprene Life Vest,49.98,Standard Class,06-21-2017,06-15-2017,10:18
24095,CASH,5,4,26.20,100.77,Late delivery,1,29,Shop By Sport,Hayward,EE. UU.,Alan,1347,Gillespie,Corporate,CA,6122 Easy Oak Extension,94544,5,Golf,37.637890,-122.076569,Qitaihe,China,1347,24319,627,19.20,0.16,60840,39.99,0.26,3,119.97,100.77,26.20,Eastern Asia,CLOSED,627,29,Under Armour Girls' Toddler Spine Surge Runni,39.99,Standard Class,12-26-2015,12-21-2015,23:40
142305,TRANSFER,5,4,19.99,199.92,Late delivery,1,46,Indoor/Outdoor Games,Cordova,EE. UU.,Mary,4120,Ortiz,Corporate,TN,1995 Silent Square,38018,7,Fan Shop,35.163326,-89.812386,Adelaide,Australia,4120,21874,1014,49.98,0.20,54703,49.98,0.10,5,249.90,199.92,19.99,Oceania,PROCESSING,1014,46,O'Brien Men's Neoprene Life Vest,49.98,Standard Class,11-21-2015,11-16-2015,07:05


# "product\_status", "customer\_email", "customer\_password", "product\_description", "order\_zipcode", "market", "order\_state","product\_image" "shipping\_date\_dateorders"


In [11]:
department = df_data[["department_id", "department_name"]].drop_duplicates(subset=["department_id"]).reset_index(drop=True)

In [12]:
department.sample(5)

,department_id,department_name
1,4,Apparel
9,11,Pet Shop
5,7,Fan Shop
2,5,Golf
0,2,Fitness


In [13]:
product = df_data[["product_card_id", "product_name", "product_price", "category_id"]].drop_duplicates(subset=["product_card_id"]).reset_index(drop=True)

In [16]:
product.sample(5)

,product_card_id,product_name,product_price,category_id
21,60,SOLE E25 Elliptical,999.99,4
104,715,TaylorMade Women's RBZ SL Rescue,129.99,32
47,203,GoPro HERO3+ Black Edition Camera,399.99,10
19,1349,Web Camera,452.04,62
83,35,adidas Brazuca 2014 Official Match Ball,159.99,3


In [ ]:
dim_customers = df_data[['customer_id', 'customer_fname', 'customer_lname', 
                    'customer_segment', 'customer_street', 
                    'customer_city', 'customer_state', 'customer_zipcode', 'customer_country'
                    ]].drop_duplicates(subset=['customer_id']).reset_index(drop=True)

In [ ]:
dim_customers.head(5)

,customer_id,customer_fname,customer_lname,customer_segment,customer_street,customer_city,customer_state,customer_zipcode,customer_country
0,20755,Cally,Holloway,Consumer,5365 Noble Nectar Island,Caguas,PR,725,Puerto Rico
1,19492,Irene,Luna,Consumer,2679 Rustic Loop,Caguas,PR,725,Puerto Rico
2,19491,Gillian,Maldonado,Consumer,8510 Round Bear Gate,San Jose,CA,95125,EE. UU.
3,19490,Tana,Tate,Home Office,3200 Amber Bend,Los Angeles,CA,90027,EE. UU.
4,19489,Orli,Hendricks,Corporate,8671 Iron Anchor Corners,Caguas,PR,725,Puerto Rico


In [ ]:
dim_stores = df_data[['latitude', 'longitude', 'shipping_mode']].drop_duplicates().reset_index(drop=True)
dim_stores['store_id'] = dim_stores.index + 1
dim_stores = dim_stores[['store_id', 'latitude', 'longitude', 'shipping_mode']]

In [ ]:
dim_stores.sample(5)

,store_id,latitude,longitude,shipping_mode
12911,12912,18.233334,-66.370529,Standard Class
1942,1943,18.237600,-66.370506,Second Class
10557,10558,18.245821,-66.370567,Standard Class
15778,15779,18.245676,-66.370590,Same Day
14769,14770,18.296942,-66.370506,First Class


In [ ]:
fact_sales = pd.merge(df_data, dim_stores, on=['latitude', 'longitude', 'shipping_mode'], how='left')

In [ ]:
fact_sales.sample(5)

,type,days_for_shipping_real,days_for_shipment_scheduled,benefit_per_order,sales_per_customer,delivery_status,late_delivery_risk,category_id,category_name,customer_city,customer_country,customer_fname,customer_id,customer_lname,customer_segment,customer_state,customer_street,customer_zipcode,department_id,department_name,latitude,longitude,order_city,order_country,order_customer_id,order_id,order_item_cardprod_id,order_item_discount,order_item_discount_rate,order_item_id,order_item_product_price,order_item_profit_ratio,order_item_quantity,sales,order_item_total,order_profit_per_order,order_region,order_status,product_card_id,product_category_id,product_name,product_price,shipping_mode,delivery_date,order_date,order_time,store_id
8426,DEBIT,5,4,-4.67,122.84,Late delivery,1,18,Men's Footwear,Brooklyn,EE. UU.,Margaret,3997,Walker,Corporate,NY,5731 Cinder Horse Court,11218,4,Apparel,40.640594,-73.975189,Chattanooga,Estados Unidos,3997,40034,403,7.15,0.06,99891,129.99,-0.04,1,129.99,122.84,-4.67,South of USA,COMPLETE,403,18,Nike Men's CJ Elite 2 TD Football Cleat,129.99,Standard Class,08-12-2016,08-07-2016,09:20,300
47371,PAYMENT,4,4,64.02,176.37,Shipping on time,0,17,Cleats,Far Rockaway,EE. UU.,Mary,10164,Parsons,Corporate,NY,2 Foggy Corners,11691,4,Apparel,42.088577,-76.888535,Ulan Bator,Mongolia,10164,45513,365,3.60,0.02,113736,59.99,0.36,3,179.97,176.37,64.02,Eastern Asia,PENDING_PAYMENT,365,17,Perfect Fitness Perfect Rip Deck,59.99,Standard Class,10-30-2016,10-26-2016,08:52,3187
40263,DEBIT,6,4,10.40,129.99,Late delivery,1,18,Men's Footwear,Caguas,Puerto Rico,Eric,2833,Vega,Corporate,PR,4377 Heather Canyon,725,4,Apparel,18.203897,-66.370544,Cairo,Egipto,2833,51070,403,0.00,0.00,127645,129.99,0.08,1,129.99,129.99,10.40,North Africa,COMPLETE,403,18,Nike Men's CJ Elite 2 TD Football Cleat,129.99,Standard Class,01-21-2017,01-15-2017,11:44,12322
56918,DEBIT,2,4,-2.29,176.37,Advance shipping,0,17,Cleats,Caguas,Puerto Rico,Mary,11305,Hartman,Consumer,PR,5193 Foggy Apple Jetty,725,4,Apparel,18.232094,-66.370590,Los Angeles,Estados Unidos,11305,37071,365,3.60,0.02,92496,59.99,-0.01,3,179.97,176.37,-2.29,West of USA,COMPLETE,365,17,Perfect Fitness Perfect Rip Deck,59.99,Standard Class,06-27-2016,06-25-2016,03:16,8094
63320,CASH,5,4,-16.63,47.50,Late delivery,1,24,Women's Apparel,Harvey,EE. UU.,David,2113,Smith,Home Office,IL,5902 Silver Embers Diversion,60426,5,Golf,41.786617,-90.132309,BerlÃ­n,Alemania,2113,13824,502,2.50,0.05,34632,50.00,-0.35,1,50.00,47.50,-16.63,Western Europe,CLOSED,502,24,Nike Men's Dri-FIT Victory Golf Polo,50.00,Standard Class,07-26-2015,07-21-2015,18:48,10715


In [ ]:
fact_sales = fact_sales[[
    'order_item_id', 'order_id', 'customer_id', 'product_card_id', 'store_id',
    'order_date_dateorders',
    'type', 'order_status', 'days_for_shipping_real', 
    'days_for_shipment_scheduled', 'delivery_status', 'late_delivery_risk',
    'order_city', 'order_country', 'order_region',
    'sales', 'order_item_quantity', 'order_item_product_price', 
    'order_item_discount', 'order_item_discount_rate', 'order_item_total', 
    'order_item_profit_ratio', 'benefit_per_order', 'order_profit_per_order', 'sales_per_customer'
]]
 
print("¡DataFrames del Modelo en Estrella listos en memoria!")

KeyError: "['order_date_dateorders'] not in index"